In [1]:
import os
from dotenv import load_dotenv
import chromadb
from chromadb.utils import embedding_functions

load_dotenv()

True

In [2]:
chroma_client=chromadb.PersistentClient(path="chroma_storage_v2")

In [3]:
local_ef=embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection=chroma_client.get_or_create_collection(name="legal_reports_v2", embedding_function=local_ef)    
print("ChromaDB V2 setup complete.")

e:\AgenticRagProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8439.74it/s]


ChromaDB V2 setup complete.


In [4]:
import os
from pypdf import PdfReader

# 1. Point to your raw documents folder
pdf_folder = "documents/pdfdata"  
sample_documents = []

# 2. Automatically find and read every PDF file
for filename in os.listdir(pdf_folder):
    if filename.lower().endswith('.pdf'):
        pdf_path = os.path.join(pdf_folder, filename)
        print(f"Extracting text from: {filename}")
        
        try:
            reader = PdfReader(pdf_path)
            full_text = ""
            for page in reader.pages:
                text_content = page.extract_text()
                if text_content:
                    full_text += text_content + "\n"
            
            # Append the real data structure to our list
            sample_documents.append({
                "source": filename,
                "text": full_text
            })
        except Exception as e:
            print(f"Error reading {filename}: {e}")

Extracting text from: Constitution_India.pdf


In [5]:
child_texts = []
child_ids = []
child_metadatas = []

PARENT_SIZE = 2000
PARENT_OVERLAP = 400
CHILD_SIZE = 400
CHILD_OVERLAP = 50

print("Beginning hierarchical slicing loop...")

for i,doc in enumerate(sample_documents):
    text = doc["text"]
    source_file = doc["source"]
    
    # --- STEP 1: Slice Parent Windows ---
    parent_index = 0
    start_p = 0
    while start_p < len(text):
        end_p = min(start_p + PARENT_SIZE, len(text))
        parent_text = text[start_p:end_p]
        parent_id = f"parent_{source_file}_{parent_index}"
        
        # --- STEP 2: Slice Child Windows inside this Parent ---
        child_index = 0
        start_c = 0
        while start_c < len(parent_text):
            end_c = min(start_c + CHILD_SIZE, len(parent_text))
            child_text = parent_text[start_c:end_c]
            
            # Ensure we don't grab empty string scraps at the very end
            if len(child_text.strip()) > 10:
                child_id = f"child_{parent_id}_{child_index}"
                
                # --- STEP 3: Map Metadata ---
                # We inject the massive parent text directly inside the child's metadata
                metadata = {
                    "source": source_file,
                    "parent_id": parent_id,
                    "parent_text": parent_text
                }
                
                child_texts.append(child_text)
                child_ids.append(child_id)
                child_metadatas.append(metadata)
                
            child_index += 1
            start_c += (CHILD_SIZE - CHILD_OVERLAP)
            if i%50==0:
                print(f"Slicing Child {child_index} in Parent {parent_index} of {source_file}")
            
        parent_index += 1
        start_p += (PARENT_SIZE - PARENT_OVERLAP)

print(f"Slicing Complete. Generated {len(child_texts)} unique child chunks mapped to parent frameworks.")

Beginning hierarchical slicing loop...
Slicing Child 1 in Parent 0 of Constitution_India.pdf
Slicing Child 2 in Parent 0 of Constitution_India.pdf
Slicing Child 3 in Parent 0 of Constitution_India.pdf
Slicing Child 4 in Parent 0 of Constitution_India.pdf
Slicing Child 5 in Parent 0 of Constitution_India.pdf
Slicing Child 6 in Parent 0 of Constitution_India.pdf
Slicing Child 1 in Parent 1 of Constitution_India.pdf
Slicing Child 2 in Parent 1 of Constitution_India.pdf
Slicing Child 3 in Parent 1 of Constitution_India.pdf
Slicing Child 4 in Parent 1 of Constitution_India.pdf
Slicing Child 5 in Parent 1 of Constitution_India.pdf
Slicing Child 6 in Parent 1 of Constitution_India.pdf
Slicing Child 1 in Parent 2 of Constitution_India.pdf
Slicing Child 2 in Parent 2 of Constitution_India.pdf
Slicing Child 3 in Parent 2 of Constitution_India.pdf
Slicing Child 4 in Parent 2 of Constitution_India.pdf
Slicing Child 5 in Parent 2 of Constitution_India.pdf
Slicing Child 6 in Parent 2 of Constitution

In [6]:
# Batch data into the database
# Chroma handles the local vectorization of the child documents automatically
print("Writing hierarchical index to chroma_storage_v2...")

# Chroma has a batch limit default, if your dataset is huge we slice it, otherwise direct add works:
BATCH_SIZE = 1000
for i in range(0, len(child_texts), BATCH_SIZE):
    collection.add(
        documents=child_texts[i:i+BATCH_SIZE],
        ids=child_ids[i:i+BATCH_SIZE],
        metadatas=child_metadatas[i:i+BATCH_SIZE]
    )

print(f" Success! Database built successfully with {collection.count()} total items.")

Writing hierarchical index to chroma_storage_v2...
 Success! Database built successfully with 3198 total items.


In [ ]:
# Run a quick validation query
test_query = "Fundamental duties of individuals in India"
results = collection.query(query_texts=[test_query], n_results=1)

if results['documents'][0]:
    print("TEST RETRIEVAL SUCCESSFUL\n" + "="*50)
    print(f"Matched Child Segment:\n{results['documents'][0][0]}")
    print("-"*50)
    print(f"Linked Parent Text Window (Extracted from Metadata):\n{results['metadatas'][0][0]['parent_text'][:500]}...")
    print("="*50)
    print("Parent-Child structural mapping is verified and functioning flawlessly.")
else:
    print("Query returned empty. Double check your document text parsing variables.")

TEST RETRIEVAL SUCCESSFUL
Matched Child Segment:
5
1[PART  IVAFUNDAMENTAL DUTIES 51A. Fundamental duties.—It shall be the duty of every citizen of India—(a) to abide by the Constitution and respect its ideals and institutions, the National Flag and the National Anthem;(b) to cherish and follow the noble ideals which inspired our national struggle for freedom; (c) to uphold and protect the sovereignty, unity and integrity of India;(d) to defend t
--------------------------------------------------
Linked Parent Text Window (Extracted from Metadata):
 respect for international law and treaty obligations in the dealings of organised peoples with one another; and(d) encourage settlement of international disputes by arbitration.  ______________________________________________1. Subs. by the Constitution (Seventh Amendment) Act, 1956, s. 27, for "declared by Parliament by law" (w.e.f. 1-11-1956).
25
1[PART  IVAFUNDAMENTAL DUTIES 51A. Fundamental duties.—It shall be the duty of every citizen o

: 